# X5: original XGBoost on CUDA
Private reproduction; validation and final forecast only. No competition submission.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'xgboost==3.1.3', 'pandas==2.2.3', 'numpy==2.3.5', 'pyarrow==26.0.0', 'holidays==0.106'])


In [ ]:
from pathlib import Path
runner = Path('/kaggle/working/gpu_run.py')
runner.write_text('"""Kaggle entry point: run only the original XGBoost validation and final fits."""\nfrom __future__ import annotations\n\nimport ast\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport subprocess\nfrom pathlib import Path\n\nimport holidays\nimport numpy as np\nimport pandas as pd\nimport xgboost\n\nEXPECTED_EFFECTIVE_TARGETS = {\'ytrain\': {\'columns\': [\'target\'], \'dtypes\': [\'float32\'], \'rows\': 466425, \'sha256\': \'7b442f87c577fd2823f27e450af5331f9925853abdce62521d6370bb9bced0c2\'}, \'yholdout\': {\'columns\': [\'target\'], \'dtypes\': [\'float32\'], \'rows\': 18657, \'sha256\': \'7cae9959bb3577d86cbaa61417d70dc548b5ace56d5b947a1421bf579c4fe253\'}}\n\n\ndef fingerprint(frame):\n    return dict(columns=list(frame.columns), dtypes=[str(t) for t in frame.dtypes],\n                rows=len(frame), sha256=hashlib.sha256(\n                    pd.util.hash_pandas_object(frame, index=False, categorize=False).values.tobytes()).hexdigest())\n\n\ndef main():\n    inputs = list(Path(\'/kaggle/input\').rglob(\'input_manifest.json\'))\n    assert len(inputs) == 1, \'Expected exactly one reproduction input package\'\n    source = inputs[0].parent\n    out = Path(\'/kaggle/working\')\n    manifest = json.loads(inputs[0].read_text())\n    for name, digest in manifest[\'input_hashes\'].items():\n        assert hashlib.sha256((source / name).read_bytes()).hexdigest() == digest, name\n    hardware = subprocess.check_output([\'nvidia-smi\'], text=True)\n    runtime = dict(python=platform.python_version(), gpu=hardware,\n                   versions={name: importlib.metadata.version(name) for name in\n                             [\'xgboost\', \'pandas\', \'numpy\', \'pyarrow\', \'holidays\']},\n                   input_manifest=manifest)\n    (out / \'runtime.json\').write_text(json.dumps(runtime, indent=2))\n    print(\'RUNTIME\', runtime[\'python\'], runtime[\'versions\'], hardware, flush=True)\n    notebook = json.loads((source / \'original_solution.ipynb\').read_text())\n    namespace = dict(holidays=holidays)\n    # Reuse code definitions and parameter registries only. Installation cells,\n    # markdown and stored outputs from the provided notebook are never executed.\n    for index in [10, 14, 16, 37]:\n        exec(compile(\'\'.join(notebook[\'cells\'][index][\'source\']),\n                     f\'original_solution:cell_{index}\', \'exec\'), namespace)\n    rename = ast.literal_eval(ast.parse(\'\'.join(notebook[\'cells\'][8][\'source\'])).body[0].value)\n    for node in ast.parse(\'\'.join(notebook[\'cells\'][18][\'source\'])).body:\n        if isinstance(node, ast.FunctionDef):\n            exec(compile(ast.Module(body=[node], type_ignores=[]), \'original_preparation\', \'exec\'), namespace)\n        elif isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name):\n            if node.targets[0].id in {\'COMMON_BASE_FEATURE_KWARGS\', \'ID_COL\', \'YEAR_COL\',\n                                      \'MONTH_COL\', \'TIME_COL\', \'TARGET_COL\'}:\n                namespace[node.targets[0].id] = ast.literal_eval(node.value)\n    raw = pd.read_csv(source / \'train_2.csv\').rename(columns=rename)\n    assert len(raw) == 485082 and raw.new_id.nunique() == 18657\n    with __import__(\'warnings\').catch_warnings():\n        __import__(\'warnings\').simplefilter(\'ignore\', pd.errors.PerformanceWarning)\n        features = namespace[\'build_retail_features_with_split\'](\n            raw, split_mode=\'train_val_test\', min_year=2023,\n            val_year=2025, val_month=2, test_year=2025, test_month=3,\n            create_test_if_missing=True, expected_test_size=18657,\n            use_val_as_test_history=True,\n            base_feature_kwargs=namespace[\'COMMON_BASE_FEATURE_KWARGS\'], verbose=True)\n    del raw\n    gc.collect()\n    selected = namespace[\'xgboost_log_diff_features\']\n    pack = namespace[\'_prepare_numeric_model_pack\'](\n        features, selected_features=selected, target_mode=\'growth_log\',\n        label=\'xgboost_log_diff\', verbose=True)\n    validation, final = pack[\'validation\'], pack[\'test\']\n    expected = json.loads((source / \'expected_validation_features.json\').read_text())\n    actual = {name: fingerprint(frame) for name, frame in {\n        \'train\': validation[\'X_train\'][selected], \'holdout\': validation[\'X_val\'][selected],\n        \'ytrain\': pd.DataFrame({\'target\': validation[\'y_train\']}),\n        \'yholdout\': pd.DataFrame({\'target\': validation[\'y_val\']})}.items()}\n    # Linux and Windows may differ in the last float64 bits of np.log. XGBoost\n    # stores labels in float32; compare the effective finite values and drop mask.\n    effective = {name: fingerprint(frame.astype(\'float32\').where(np.isfinite(frame), np.nan))\n                 for name, frame in {\'ytrain\': pd.DataFrame({\'target\': validation[\'y_train\']}),\n                                     \'yholdout\': pd.DataFrame({\'target\': validation[\'y_val\']})}.items()}\n    pd.DataFrame({\'target\': validation[\'y_train\']}).to_parquet(out / \'training_labels_float64.parquet\', index=False)\n    passed = all(actual[k] == expected[k] for k in [\'train\', \'holdout\']) and effective == EXPECTED_EFFECTIVE_TARGETS\n    (out / \'feature_checks.json\').write_text(json.dumps(dict(expected=expected, actual=actual,\n        effective_targets_expected=EXPECTED_EFFECTIVE_TARGETS, effective_targets_actual=effective,\n        passed=passed), indent=2))\n    assert passed, \'GPU features or effective labels differ from the reproduced local baseline\'\n    print(\'FEATURE_PARITY_PASSED\', flush=True)\n    # Release unselected columns before fitting; metadata keeps original anchors.\n    meta_columns = [\'new_id\', \'year\', \'month\', \'year_month_index\', \'region\', \'settlement\',\n                    \'pto\', \'pto_lag_1\', \'opening_date_category\', \'trading_area_category\',\n                    \'population_city_type\', \'lag1_rto_size_bin\', \'month_name\', \'holiday_month_type\']\n    val_meta = features[\'val_features\'][[c for c in meta_columns if c in features[\'val_features\']]].copy()\n    test_meta = features[\'test_features\'][[c for c in meta_columns if c in features[\'test_features\']]].copy()\n    del features\n    gc.collect()\n    params = dict(namespace[\'xgboost_log_diff_params\'])\n    assert params[\'device\'] == \'cuda\' and params[\'n_estimators\'] == 3750\n    result = namespace[\'run_xgboost_log_diff\'](\n        params=params, features=selected,\n        val_train_df=validation[\'X_train\'], val_test_df=validation[\'X_val\'],\n        val_y_train=validation[\'y_train\'], val_y_test=validation[\'y_val\'],\n        full_train_df=final[\'X_train\'], full_test_df=final[\'X_test\'], full_y_train=final[\'y_train\'],\n        val_test_meta_df=val_meta, full_test_meta_df=test_meta,\n        output_dir=str(out / \'outputs_xgboost_log_diff\'),\n        submission_filename=\'test_xgboost_log_diff.csv\',\n        final_iterations_strategy=\'params\', save_files=True, verbose=True)\n    configuration = json.loads(result[\'val_model\'].get_booster().save_config())\n    actual_device = configuration[\'learner\'][\'generic_param\'][\'device\']\n    assert actual_device.startswith(\'cuda\'), \'XGBoost silently fell back to CPU\'\n    metrics = result[\'val_metrics\']\n    metrics.update(device=actual_device, notebook_sha256=manifest[\'input_hashes\'][\'original_solution.ipynb\'],\n                   original_saved_mape=4.4314600477414245,\n                   delta_from_saved_mape=metrics[\'mape\'] - 4.4314600477414245,\n                   runtime_versions=runtime[\'versions\'])\n    (out / \'gpu_metrics.json\').write_text(json.dumps(metrics, indent=2))\n    for label in [\'val\', \'full\']:\n        result[label + \'_model\'].save_model(out / (label + \'_model.ubj\'))\n    print(\'GPU_RESULT\', json.dumps(metrics), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
subprocess.check_call([sys.executable, '-u', str(runner)])
